# L18: RAG与向量数据库


# L18: RAG与向量数据库

**课时**: 2 课时（90 分钟/课时）

## 1. 学习目标

完成本课程后，学员能够：
1. 解释RAG（检索增强生成）的核心原理与为什么能缓解幻觉问题
2. 理解文本Embedding的原理（Word2Vec、BERT、Sentence-BERT）
3. 掌握向量数据库的核心概念：向量索引、相似度度量、ANN算法
4. 使用Chroma/LangChain等工具实现完整的RAG Pipeline
5. 评估RAG系统的性能并优化检索质量

## 2. 核心知识点

### 2.1 RAG（检索增强生成）概述

**为什么需要RAG**:
- LLM知识有截止日期，无法获取最新信息
- LLM会产生幻觉，生成看似合理但错误的内容
- 无法提供具体的、专有的领域知识

**RAG工作流程**:


In [ ]:
用户查询 → 编码为向量 → 向量数据库检索 → 获取相关文档 → 
拼接到提示词 → LLM生成答案 → 返回最终结果


**RAG vs Fine-tuning**:
| 方面 | RAG | Fine-tuning |
|------|-----|-------------|
| 更新知识 | 快（新文档即时可用） | 慢（需要重新训练） |
| 成本 | 低（无需训练） | 高（需要GPU训练） |
| 可解释性 | 高（可追溯检索来源） | 低（隐式知识） |
| 幻觉减少 | 强（基于检索证据） | 中（取决于训练数据） |
| 适用场景 | 知识库、文档问答 | 风格迁移、任务适配 |

### 2.2 文本 Embedding

**词向量**: 将文本表示为稠密向量，捕捉语义信息

**Word2Vec**:
- CBOW: 根据上下文预测中心词
- Skip-gram: 根据中心词预测上下文
- 训练目标：相似词有相似向量


In [ ]:
# 使用预训练Word2Vec
from gensim.models import KeyedVectors

# 加载预训练模型（需要下载）
# model = KeyedVectors.load_word2vec_format('GoogleNews-vectors-negative300.bin', binary=True)

# 演示：词向量运算
# king - man + woman ≈ queen
# vector = model['king'] - model['man'] + model['woman']
# most_similar = model.most_similar([vector])


**BERT Embedding**:
- 双向Transformer encoder
- [CLS] token的输出作为句子表示
- 支持上下文相关的词表示


In [ ]:
from transformers import BertModel, BertTokenizer
import torch

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

text = "The cat sat on the mat."
inputs = tokenizer(text, return_tensors='pt')
outputs = model(**inputs)

# [CLS] token的输出作为句子表示
cls_embedding = outputs.last_hidden_state[:, 0, :]
print(f"Embedding维度: {cls_embedding.shape}")  # [1, 768]


**Sentence-BERT (SBERT)**:
- 针对语义相似度任务优化
- 使用孪生网络结构训练
- 输出固定维度的句子向量


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')  # 384维，最快的模型之一

sentences = [
    "A man is eating a piece of bread",
    "Someone is enjoying a meal",
    "A woman is playing the violin"
]

embeddings = model.encode(sentences)
print(f"Embedding形状: {embeddings.shape}")  # [3, 384]

# 计算相似度
from sklearn.metrics.pairwise import cosine_similarity
similarity = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]
print(f"句子1与句子2的余弦相似度: {similarity:.3f}")


### 2.3 向量数据库

**核心概念**:
- **向量索引**: 加速大规模向量检索的数据结构
- **相似度度量**: 余弦相似度、欧氏距离、点积
- **ANN (近似最近邻)**: 在可接受精度损失下大幅加速检索

**主流向量数据库对比**:

| 数据库 | 特点 | 适用场景 |
|--------|------|----------|
| Chroma | 轻量、易用、Python优先 | 原型、小规模 |
| Milvus | 分布式、高性能 | 生产环境 |
| Pinecone | 云原生、托管服务 | 企业级 |
| Weaviate | 混合搜索（向量+关键词） | 复杂检索 |
| Qdrant | Rust实现、高性能 | 自托管 |

**索引算法**:
- **HNSW (Hierarchical Navigable Small World)**: 分层图索引，召回率高但内存占用大
- **IVF (Inverted File Index)**: 基于聚类的索引，平衡速度与精度
- **PQ (Product Quantization)**: 向量压缩，大幅降低内存但精度下降


In [ ]:
import chromadb

# 创建客户端（内存模式，适合开发）
client = chromadb.Client()

# 创建集合（相当于表）
collection = client.create_collection(
    name="documents",
    metadata={"hnsw:space": "cosine"}  # 使用余弦相似度
)

# 添加文档
collection.add(
    documents=[
        "深度学习是机器学习的一个分支，使用神经网络模型。",
        "自然语言处理是AI的一个领域，专注于处理人类语言。",
        "向量数据库用于存储和检索高维向量数据。"
    ],
    ids=["doc1", "doc2", "doc3"],
    metadatas=[{"source": "教科书"}, {"source": "论文"}, {"source": "技术文档"}]
)

# 查询
results = collection.query(
    query_texts=["什么是深度学习？"],
    n_results=2
)

print(f"检索到的文档: {results['documents']}")
print(f"距离: {results['distances']}")


### 2.4 完整RAG实现


In [ ]:
"""
L18-rag-pipeline.py
完整RAG系统实现：从文档到答案
"""

from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import Chroma
from langchain.chat_models import ChatOpenAI
from langchain.chains import RetrievalQA

# ============ 1. 文档加载与分块 ============
# 假设有一个文本文件
# loader = TextLoader("document.txt")
# documents = loader.load()

# 演示用示例文档
documents = [
    "人工智能（AI）是计算机科学的一个分支，旨在创造能够模拟人类智能的机器。",
    "机器学习是AI的一个子领域，通过数据学习模式而非显式编程。",
    "深度学习使用多层神经网络，在图像识别和NLP领域取得突破性进展。",
    "Transformer架构是现代LLM的基础，使用自注意力机制处理序列数据。",
    "ChatGPT是OpenAI开发的对话AI，基于GPT-3.5/4架构，通过RLHF训练。"
]

# 分块处理
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,  # 每块最大字符数
    chunk_overlap=20,  # 块之间重叠，防止句子被切断
    length_function=len
)
chunks = text_splitter.split_text(" ".join(documents))
print(f"分块数量: {len(chunks)}")

# ============ 2. 向量化存储 ============
# 使用OpenAI Embedding（需要API密钥）
# embeddings = OpenAIEmbeddings()
# vectorstore = Chroma.from_texts(chunks, embeddings, persist_directory="./db")
# vectorstore.persist()

# 演示：使用本地嵌入模型
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# 手动模拟Chroma操作
chunk_vectors = embedding_model.encode(chunks)
print(f"向量维度: {chunk_vectors.shape}")

# ============ 3. 检索 ============
query = "什么是深度学习？"
query_vector = embedding_model.encode([query])[0]

# 计算相似度
from sklearn.metrics.pairwise import cosine_similarity
similarities = cosine_similarity([query_vector], chunk_vectors)[0]

# 取Top-K
top_k = 2
top_indices = similarities.argsort()[-top_k:][::-1]
print(f"\n查询: {query}")
print(f"检索到 {top_k} 个相关块:")
for idx in top_indices:
    print(f"  - [{similarities[idx]:.3f}] {chunks[idx]}")

# ============ 4. 生成（模拟） ============
# 构建提示词
retrieved_context = "\n".join([chunks[i] for i in top_indices])
prompt = f"""
基于以下参考信息回答问题。如果信息不足，请说明不知道。

参考信息：
{retrieved_context}

问题：{query}

答案：
"""
print(f"\n最终提示词:\n{prompt}")

# 实际使用时调用LLM API
# llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
# chain = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=vectorstore.as_retriever())
# answer = chain.run(query)


## 3. 代码示例


In [ ]:
"""
L18-advanced-rag.py
高级RAG技巧：混合检索、重排序、多跳推理
"""

from sentence_transformers import SentenceTransformer
import numpy as np

# ============ 1. 混合检索（向量 + 关键词） ============
def hybrid_search(query, vector_chunks, text_chunks, top_k=3, alpha=0.5):
    """
    混合检索：结合向量相似度和BM25关键词匹配
    alpha: 向量权重（1-alpha为关键词权重）
    """
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    
    # 向量检索分数
    query_vec = embedding_model.encode([query])[0]
    vector_scores = cosine_similarity([query_vec], vector_chunks)[0]
    
    # BM25关键词检索
    vectorizer = TfidfVectorizer()
    all_texts = text_chunks + [query]
    tfidf_matrix = vectorizer.fit_transform(all_texts)
    
    query_tfidf = tfidf_matrix[-1]
    doc_tfidf = tfidf_matrix[:-1]
    bm25_scores = np.array([
        compute_bm25(query_tfidf.toarray()[0], doc_tfidf.toarray()[i]) 
        for i in range(len(text_chunks))
    ])
    
    # 归一化
    vector_scores_norm = (vector_scores - vector_scores.min()) / (vector_scores.max() - vector_scores.min() + 1e-8)
    bm25_scores_norm = (bm25_scores - bm25_scores.min()) / (bm25_scores.max() - bm25_scores.min() + 1e-8)
    
    # 混合分数
    hybrid_scores = alpha * vector_scores_norm + (1 - alpha) * bm25_scores_norm
    
    top_indices = hybrid_scores.argsort()[-top_k:][::-1]
    return [(i, hybrid_scores[i]) for i in top_indices]

# ============ 2. 重排序（Cross-Encoder） ============
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def rerank(query, candidates, top_n=3):
    """
    使用Cross-Encoder对候选文档进行重排序
    Cross-Encoder比Bi-Encoder精度更高但速度更慢
    """
    pairs = [(query, doc) for doc in candidates]
    scores = cross_encoder.predict(pairs)
    
    sorted_indices = scores.argsort()[::-1]
    return [(candidates[i], scores[i]) for i in sorted_indices[:top_n]]

# ============ 3. 多跳推理 ============
def multi_hop_rag(query, vector_chunks, text_chunks, n_hops=2):
    """
    多跳RAG：逐步检索和推理
    适用于需要综合多处信息的问题
    """
    context = []
    current_query = query
    
    for hop in range(n_hops):
        print(f"\n第 {hop+1} 跳检索: {current_query}")
        
        # 检索
        top_indices = hybrid_search(current_query, vector_chunks, text_chunks, top_k=2, alpha=0.7)
        retrieved = [text_chunks[i] for i, _ in top_indices]
        
        # 追加到上下文
        context.extend(retrieved)
        
        # 构造下一跳查询（基于已收集的信息）
        if hop < n_hops - 1:
            synthesis_prompt = f"""
根据以下信息，构造一个追问题来获取更多细节：

已收集信息：
{chr(10).join(retrieved)}

原问题：{query}

追问题（简短、具体）：
"""
            # 这里应该调用LLM生成追问题
            # current_query = llm.predict(synthesis_prompt)
            current_query = f"更多关于{retrieved[0][:20]}的细节"  # 模拟
    
    return context

# ============ 演示 ============
print("高级RAG技巧演示")
print("=" * 50)
print("1. 混合检索：结合向量和关键词搜索")
print("2. 重排序：使用Cross-Encoder提升精度")
print("3. 多跳推理：逐步获取复杂问题的答案")
print("\n提示：实际使用需要完整的向量数据库和LLM API配置")


## 4. 练习题

1. **原理解释**: 画图说明RAG的工作流程，解释为什么RAG能有效减少幻觉问题。
2. **Embedding对比**: 比较Word2Vec、BERT、SBERT三种Embedding方法的优缺点，分析各自适合的场景。
3. **向量数据库实验**: 使用Chroma或FAISS构建一个本地向量数据库，存储至少100个文档，实现相似文档检索，并比较HNSW和IVF索引的性能差异。
4. **RAG实现**: 选择一个你熟悉的领域（如技术文档、小说、新闻），构建完整的RAG系统：文档加载→分块→向量化→检索→生成，并评估回答质量。
5. **优化RAG**: 分析现有RAG系统的瓶颈，提出至少3种优化方案（提示词压缩、混合检索、重排序等），并实施其中一种验证效果。

## 5. 延伸阅读

- **论文**: Lewis et al., 2020, "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks" — RAG原始论文
- **网站**: LangChain文档 — https://python.langchain.com/
- **网站**: Chroma文档 — https://docs.trychroma.com/
- **网站**: Weaviate RAG教程 — https://weaviate.io/developers/weaviate/tutorials/rag
- **工具**: FAISS (Facebook AI Similarity Search) — https://github.com/facebookresearch/faiss
- **工具**: Sentence Transformers — https://www.sbert.net/

---
